# A floresta da família Silva

**Fonte:** Arce e Dobner (2024), transcrito em Arce, J. E. (2025). *Programação Linear para Fins Florestais*. Apostila da disciplina AS062, UFPR. p. 14–16.

**Neste notebook:** o mesmo problema da planilha `01_familia_silva.xlsx`, agora escrito em Pyomo. As duas variáveis são escritas uma a uma.

**Sua tarefa:** completar as restrições do modelo. Onde aparecer `____`, escreva o número certo (por exemplo, `10`) ou o nome do dado (por exemplo, `mao_obra_pinus`); os dois funcionam. A primeira restrição vem pronta, como exemplo.

**Se aparecer `NameError: name '____' is not defined`**, ainda falta preencher alguma lacuna.

**Como rodar:** menu **Ambiente de execução → Executar tudo**. Os passos são os mesmos da planilha:
**estimar → modelar → resolver → comparar**.

## 0. Preparar o ambiente

Instala o Pyomo (a linguagem do modelo) e o HiGHS (o *solver*). O Colab apaga tudo ao fechar a aba,
então esta célula roda sempre que o notebook é aberto.

In [ ]:
%pip install -q pyomo highspy

Abaixo ficam três ferramentas para resolver e ler o resultado em tabelas.
**Você não precisa entender este código:** basta rodar.

In [ ]:
import pyomo.environ as pyo
import pandas as pd


def br(numero, casas=2):
    """Escreve um número no formato brasileiro: 1.234,56."""
    texto = f"{numero:,.{casas}f}"
    return texto.replace(",", "X").replace(".", ",").replace("X", ".")


def resolver(modelo):
    """Chama o HiGHS, mostra o status e devolve o valor do objetivo."""
    resultado = pyo.SolverFactory("appsi_highs").solve(modelo)
    print("Status:", resultado.solver.termination_condition)
    objetivo = next(modelo.component_objects(pyo.Objective, active=True))
    print("Objetivo:", br(pyo.value(objetivo)))
    return pyo.value(objetivo)


def ler_variaveis(modelo):
    """Tabela com o valor de cada variável de decisão."""
    linhas = []
    for variavel in modelo.component_objects(pyo.Var, active=True):
        for indice in variavel:
            nome = variavel.name if indice is None else f"{variavel.name}[{indice}]"
            linhas.append({"variável": nome, "valor": round(pyo.value(variavel[indice]), 2)})
    return pd.DataFrame(linhas)


def ler_restricoes(modelo):
    """Tabela com o uso, o limite e a folga de cada restrição (como a aba 4_comparar)."""
    linhas = []
    for restricao in modelo.component_objects(pyo.Constraint, active=True):
        for indice in restricao:
            r = restricao[indice]
            usado = pyo.value(r.body)
            if r.equality:
                sinal, limite, folga = "=", pyo.value(r.upper), usado - pyo.value(r.upper)
            elif r.has_ub():
                sinal, limite, folga = "<=", pyo.value(r.upper), pyo.value(r.upper) - usado
            else:
                sinal, limite, folga = ">=", pyo.value(r.lower), usado - pyo.value(r.lower)
            nome = restricao.name if indice is None else f"{restricao.name}[{indice}]"
            linhas.append({
                "restrição": nome,
                "usado": round(usado, 2),
                "sinal": sinal,
                "limite": round(limite, 2),
                "folga": round(folga, 2),
                "limitou?": "Sim" if abs(folga) < 1e-6 else "Não",
            })
    return pd.DataFrame(linhas)

## Enunciado

Maria e José Silva têm 18 ha de floresta: 10 ha de Pinus (talhão 14) e 8 ha de Araucária (talhão 13).
Com a idade, o casal decidiu trabalhar no máximo 180 dias por ano na floresta. O trator velho foi
descartado: agora usam o trator de um vizinho, disponível até 5 dias por mês.
Cada hectare manejado pede diárias do casal e do trator, e gera uma receita por ano.

| Item | Pinus | Araucária | Limite por ano |
|---|---:|---:|---:|
| Área disponível (ha) | 10 | 8 | |
| Receita (R$/ha/ano) | 1.000 | 1.600 | |
| Mão de obra (dias/ha/ano) | 10 | 20 | 180 |
| Trator (dias/ha/ano) | 5 | 4 | 60 (5 dias/mês × 12) |

**A pergunta do casal:** quantos hectares de cada floresta manejar para ter a maior receita possível?

## Dados

Os números do enunciado, cada um com nome e unidade.

In [ ]:
# Área disponível (ha)
area_pinus = 10
area_araucaria = 8

# Receita (R$/ha/ano)
receita_pinus = 1000
receita_araucaria = 1600

# Mão de obra (dias/ha/ano) e limite do casal (dias/ano)
mao_obra_pinus = 10
mao_obra_araucaria = 20
dias_casal_ano = 180

# Trator (dias/ha/ano) e limite do vizinho: 5 dias/mês × 12 meses
trator_pinus = 5
trator_araucaria = 4
dias_trator_ano = 5 * 12

## 1. Estimar antes de modelar

Se o casal manejasse toda a área, quanto receberia? Esse valor é um **teto**: o ótimo não pode passar dele.

In [ ]:
# Teto: toda a área manejada, sem olhar mão de obra nem trator
teto = area_pinus * receita_pinus + area_araucaria * receita_araucaria
print("Teto da receita: R$", br(teto, 0), "por ano")

## 2. Modelar

$$
\begin{aligned}
\max \quad & z = 1.000\,x_1 + 1.600\,x_2 && \text{(receita, R\$/ano)}\\
\text{sujeito a} \quad & x_1 \le 10 && \text{(área de Pinus, ha)}\\
& x_2 \le 8 && \text{(área de Araucária, ha)}\\
& 10\,x_1 + 20\,x_2 \le 180 && \text{(mão de obra, dias/ano)}\\
& 5\,x_1 + 4\,x_2 \le 60 && \text{(trator, dias/ano)}\\
& x_1,\ x_2 \ge 0
\end{aligned}
$$

$x_1$ = hectares de Pinus manejados; $x_2$ = hectares de Araucária manejados.
Cada bloco de código abaixo é uma linha do modelo.

**Variáveis de decisão:** $x_1, x_2 \ge 0$ (ha).

In [ ]:
modelo = pyo.ConcreteModel("familia_silva")

# Hectares manejados de cada floresta (não negativos)
modelo.x_pinus = pyo.Var(domain=pyo.NonNegativeReals)
modelo.x_araucaria = pyo.Var(domain=pyo.NonNegativeReals)

**Objetivo:** $\max\ z = 1.000\,x_1 + 1.600\,x_2$

In [ ]:
@modelo.Objective(sense=pyo.maximize)
def receita(m):
    return receita_pinus * m.x_pinus + receita_araucaria * m.x_araucaria

**Área de Pinus:** $x_1 \le 10$ (exemplo pronto)

In [ ]:
@modelo.Constraint()
def area_pinus_ha(m):
    return m.x_pinus <= area_pinus

**Área de Araucária:** $x_2 \le 8$

In [ ]:
@modelo.Constraint()
def area_araucaria_ha(m):
    return m.x_araucaria <= ____

**Mão de obra:** $10\,x_1 + 20\,x_2 \le 180$

In [ ]:
@modelo.Constraint()
def mao_obra(m):
    return ____ * m.x_pinus + ____ * m.x_araucaria <= ____

**Trator:** $5\,x_1 + 4\,x_2 \le 60$

In [ ]:
@modelo.Constraint()
def trator(m):
    return ____ * m.x_pinus + ____ * m.x_araucaria <= ____

## 3. Resolver

In [ ]:
otimo = resolver(modelo)

In [ ]:
ler_variaveis(modelo)

## 4. Comparar

O ótimo respeita o teto? Quais recursos limitaram a receita?

In [ ]:
print("Teto (estimativa):  R$", br(teto, 0))
print("Ótimo (HiGHS):      R$", br(otimo, 0))
print("Diferença:          R$", br(teto - otimo, 0))

In [ ]:
ler_restricoes(modelo)

## 5. Perguntas

Dê dois cliques nesta célula para escrever as suas respostas.

**1. O ótimo ficou abaixo do teto da etapa 1? Por quê?**

> Sua resposta:

**2. Quais recursos limitaram a receita? Qual sobrou?**

> Sua resposta:

**3. Por que o casal não maneja toda a área, se cada hectare dá receita?**

> Sua resposta:

## 6. Para ir além: e se...?

**E se o vizinho emprestasse o trator por mais dias?** Em vez de testar um cenário de cada vez,
o Python testa vários de uma vez: um **loop** resolve o modelo 21 vezes, de 0 a 20 dias a mais
de trator por ano, e guarda o resultado de cada um numa tabela.

In [ ]:
cenarios = []
for dias_a_mais in range(0, 21):
    # Reescreve só a restrição do trator, com o novo limite
    limite = 5 * 12 + dias_a_mais
    modelo.trator.set_value(trator_pinus * modelo.x_pinus + trator_araucaria * modelo.x_araucaria <= limite)
    pyo.SolverFactory("appsi_highs").solve(modelo)
    cenarios.append({
        "dias a mais de trator": dias_a_mais,
        "trator (dias/ano)": limite,
        "Pinus (ha)": round(pyo.value(modelo.x_pinus), 2),
        "Araucária (ha)": round(pyo.value(modelo.x_araucaria), 2),
        "receita (R$/ano)": round(pyo.value(modelo.receita)),
    })

# Ganho de cada cenário em relação ao enunciado (0 dia a mais)
cenarios = pd.DataFrame(cenarios)
cenarios["ganho (R$/ano)"] = cenarios["receita (R$/ano)"] - cenarios["receita (R$/ano)"].iloc[0]
cenarios

**A partir de quantos dias a mais o trator deixa de ajudar? Por quê?**

> Sua resposta:

**E se o preço caísse?** O trator volta aos 5 dias/mês, e a receita da Araucária cai para R$ 1.200/ha/ano.

In [ ]:
# Devolve o trator ao enunciado (60 dias/ano)
dias_trator_ano = 5 * 12
modelo.trator.set_value(trator_pinus * modelo.x_pinus + trator_araucaria * modelo.x_araucaria <= dias_trator_ano)

# Nova receita da Araucária (R$/ha/ano): complete a lacuna
receita_araucaria = ____

# Reescreve só o objetivo, com a nova receita da Araucária
modelo.receita.set_value(receita_pinus * modelo.x_pinus + receita_araucaria * modelo.x_araucaria)

otimo_araucaria = resolver(modelo)
ler_variaveis(modelo)

**O que mudou?** Por que o plano mudou (ou não mudou)?